## Подготовка

Импортируем всё необходимое:

In [1]:
import os
import re
import gc
from tqdm import tqdm
import pickle
import ast
from dataclasses import dataclass, field
    
import pandas as pd
import torchaudio
import IPython
import torch

In [2]:
tqdm.pandas()

In [3]:
from TTS.api import TTS
from TTS.tts.configs.shared_configs import BaseDatasetConfig, BaseAudioConfig

C:\python_venv\venv1_py12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.tacotron2_config import Tacotron2Config
from TTS.tts.configs.shared_configs import CharactersConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.tacotron2 import Tacotron2
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Загрузим датасет

In [99]:
df = pd.read_csv("data/metadata_RUSLAN_22200.csv", delimiter='|', header=None)
df.columns = ["wav_name", "word"]
df

,wav_name,word
0,000000_RUSLAN,С тревожным чувством берусь я за перо.
1,000001_RUSLAN,Кого интересуют признания литературного неудач...
2,000002_RUSLAN,Что поучительного в его исповеди?
3,000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.
4,000004_RUSLAN,Я абсолютно здоров.
...,...,...
22195,022195_RUSLAN,"Мы жаждем совершенства, а вокруг торжествует п..."
22196,022196_RUSLAN,Революционер делает попытки установить мировую...
22197,022197_RUSLAN,"Он начинает преобразовывать жизнь, достигая ин..."
22198,022198_RUSLAN,"Допустим, выводит морковь, совершенно неотличи..."


Преобразуем данные к частоте 16 кГц:

In [100]:
DATA_DIR = "data\\RUSLAN"

TARGET_SR = 16000

In [101]:
for file in tqdm(os.listdir(DATA_DIR)):
    file_path = os.path.join(DATA_DIR, file)
    wav, sr = torchaudio.load(file_path)
    if sr == TARGET_SR:
        continue
        
    wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    torchaudio.save(file_path, wav, TARGET_SR)

  0%|                                                                                        | 0/22200 [00:00<?, ?it/s]C:\python_venv\venv1_py12\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
100%|██████████████████████████████████████████████████████████████████████████| 22200/22200 [00:13<00:00, 1695.13it/s]


Уберём все записи длиннее 10 секунд из данных:

In [6]:
def check_length(wav_name, max_seconds_length):
    file_path = os.path.join(DATA_DIR, wav_name)
    wav, sr = torchaudio.load(file_path)
    return len(wav.flatten()) / sr < max_seconds_length

In [103]:
df = df[df["wav_name"].progress_apply(lambda x: check_length(x + ".wav", 10))]

100%|██████████████████████████████████████████████████████████████████████████| 22200/22200 [00:12<00:00, 1805.08it/s]


Нормализуем тексты:

In [7]:
def is_normalized_text(text):
    if not isinstance(text, str) or text.strip() == '':
        return False
    
    patterns = [
        # Сокращения с точками
        r'\b[а-я]{1,2}\.[а-я]{1,2}\.',
        
        # Инициалы
        r'\b[А-Я]\.\s*[А-Я]\.',
        r'\b[А-Я]\.[А-Я]\.',
        
        # Аббревиатуры из заглавных букв
        r'\b[А-Я]{2,}\b',
        
        # Числа
        r'\d+[\d,.]*\d*',
        
        # Специальные символы (/, №, # и др.)
        r'[/\\№#@]'
    ]
    
    for pattern in patterns:
        if re.search(pattern, text):
            return False
    
    return True

In [105]:
df = df[df["word"].apply(is_normalized_text)]
df["word"] = df["word"].apply(lambda x: re.sub("[^а-яА-Я,.!?]", " ", x))
df["word"] = df["word"].apply(lambda x: re.sub(" +", " " , x.lower()).split())
len(df)

21374

Отделим слова от знаков препинания:

In [106]:
df = df.explode("word")

In [8]:
def split_letters_and_symbols(word):
    symbols = ""
    for i in range(len(word) - 1, -1, -1):
        if word[i].isalpha():
            return word[0:i + 1], symbols

        symbols = word[i] + symbols

    return "", symbols

In [108]:
df["punctuation"] = df["word"].apply(lambda x: split_letters_and_symbols(x)[1])

In [109]:
df["word"] = df["word"].apply(lambda x: split_letters_and_symbols(x)[0])

Возьмём токенизатор из предыдущей лабораторной работы:

In [9]:
class Characters():
    def __init__(self, num_chars):
        self.num_chars = num_chars

In [10]:
class TokensEncoder():
    def __init__(self):
        self.tokens_count = 4
        self.tokens_to_ids = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.pad_id = 0
        self.use_phonemes = False
        
    
    def fit(self, df_column):
        for index, row in df_column.items():
            for token in row:
                if token not in self.tokens_to_ids:
                    self.tokens_to_ids[token] = self.tokens_count
                    self.tokens_count += 1

        self.ids_to_tokens = {value: key for key, value in self.tokens_to_ids.items()}
        self.characters = Characters(self.tokens_count)

    def transform(self, df_column):
        return df_column.apply(lambda x: [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in x])

    def transform_one(self, tokens):
        return [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in tokens]

    def inverse_transform(self, df_column):
        return df_column.apply(lambda x: [self.ids_to_tokens[id] for id in x])

    def inverse_transform_one(self, ids):
        return [self.ids_to_tokens[id] for id in ids]

    def text_to_ids(self, text, language=None):
        while isinstance(text, str):
            try:
                text = pd.eval(text)
            except:
                print(text)
                raise
                

        return [self.tokens_to_ids[token] for token in text]

    def print_logs(self, log_level):
        pass

Напишем функции, добавляющие и удаляющие BOS, EOS и PAD символы:

In [11]:
def add_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(lambda x: ["<BOS>"] + x + ["<EOS>"])

In [12]:
def add_bos_eos_to_list(word):
    return ["<BOS>"] + word + ["<EOS>"]

In [13]:
def remove_bos_eos_from_list(word):
    while "<BOS>" in word:
        word.remove("<BOS>")
    
    while "<EOS>" in word:
        word.remove("<EOS>")
    
    return word

In [14]:
def remove_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(remove_bos_eos_from_list)

In [15]:
def remove_pad_from_list(word):
    while "<PAD>" in word:
        word.remove("<PAD>")
    
    return word

In [16]:
def remove_pad(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(remove_pad_from_list)

И функцию для применения токенизатора:

In [17]:
def train_and_apply_tokens_encoder(tokens_encoder, train_df, val_df, column):
    tokens_encoder.fit(train_df[column])
    train_df[column] = tokens_encoder.transform(train_df[column])
    val_df[column] = tokens_encoder.transform(val_df[column])

Модифицируем класс датасета из предыдущей лабораторной:

In [18]:
class WordsDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df
        self.words = [torch.LongTensor(x).to(device) for x in tqdm(df["word"].values)]
        self.allophones = [torch.LongTensor(x).to(device) for x in tqdm(df["allophones"].values)]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        return self.words[index], self.allophones[index]


И класс для выравнивания длины тензоров в батче:

In [19]:
class WordsPadder():
    def __init__(self, pad_token):
        self.pad_token = pad_token

    def __call__(self, batch):
        words, allophone_sequences = zip(*batch)
        
        padded_words = torch.nn.utils.rnn.pad_sequence(words, padding_value=self.pad_token)
        words_pad_mask = (padded_words == self.pad_token)
        padded_allophone_sequences = torch.nn.utils.rnn.pad_sequence(allophone_sequences, padding_value=self.pad_token)
        allophones_pad_mask = (padded_allophone_sequences == self.pad_token)
        
        return padded_words, padded_allophone_sequences, words_pad_mask.transpose(-1, -2), allophones_pad_mask.transpose(-1, -2)



Модифицируем также класс модели, переводящей последовательность графем в последовательность аллофонов, из предыдущей лабораторной работы:

In [20]:
class Seq2seq(torch.nn.Module):
    def __init__(self, word_tokens_count, allophone_tokens_count, transformer_dim=512, max_seq_len=100, dropout_prob=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.words_emb = torch.nn.Embedding(word_tokens_count, transformer_dim)
        self.words_positional_encoding = torch.nn.Embedding(self.max_seq_len, transformer_dim)
        self.allophones_emb = torch.nn.Embedding(allophone_tokens_count, transformer_dim)
        self.allophones_positional_encoding = torch.nn.Embedding(self.max_seq_len, transformer_dim)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.transformer = torch.nn.Transformer(d_model=transformer_dim, dropout=dropout_prob)
        self.linear = torch.nn.Linear(transformer_dim, allophone_tokens_count)
    
    def forward(self, words, allophones, words_pad_mask, allophones_pad_mask):
        words = self.words_emb(words)
        words += self.words_positional_encoding(torch.arange(words.shape[0]).to(device)).unsqueeze(1)
        words = self.dropout(words)
        
        allophones = self.allophones_emb(allophones)
        allophones += self.allophones_positional_encoding(torch.arange(allophones.shape[0]).to(device)).unsqueeze(1)
        allophones = self.dropout(allophones)
        result = self.transformer(words, allophones, tgt_mask=torch.nn.Transformer.generate_square_subsequent_mask(allophones.shape[0]).to(device), src_key_padding_mask=words_pad_mask, tgt_key_padding_mask=allophones_pad_mask)
        return self.linear(result)
    
    def words_to_allophones(self, word, max_length=None):
        if max_length is None:
            max_length = self.max_seq_len
        
        with torch.no_grad():
            allophones = torch.ones((1, word.size(1)), device=device).long()
            finished = torch.zeros(word.size(1), device=device).bool()

            for i in range(0, max_length):
                if finished.all():
                    break
                
                output = self.forward(word, allophones, None, (allophones == 0).transpose(-1, -2))
                
                next_tokens = output.argmax(dim=-1)[output.shape[0] - 1:]
                
                allophones = torch.cat([allophones, next_tokens], dim=0)
                
                finished = finished | (next_tokens.squeeze(0) == 2)
            
            return allophones
    
    def word_to_allophones(self, word):
        allophones = torch.ones((1, 1)).to(device).long()
        while allophones[-1, 0] != 2 and len(allophones) < self.max_seq_len:
            result = self.forward(word, allophones, None, (allophones == 0).transpose(-1, -2))
            allophones = torch.cat([allophones, result.argmax(dim=-1)[result.shape[0] - 1, :].unsqueeze(1)], dim=0)
    
        return allophones


Загрузим модель и токенизаторы, обученные в предыдущей лабораторной работе:

In [21]:
loaded_model = torch.load("model.pt", weights_only=False)
loaded_model.eval()

with open("word_tokens_encoder.pickle", "rb") as file:
    loaded_word_tokens_encoder = pickle.load(file)

with open("allophones_tokens_encoder.pickle", "rb") as file:
    loaded_allophones_tokens_encoder = pickle.load(file)

In [22]:
loaded_model = loaded_model.to(device)

Преобразуем слова в аллофоны:

In [114]:
df["word"] = df["word"].apply(list)
add_bos_eos(df, "word")
df["word"] = loaded_word_tokens_encoder.transform(df["word"])

In [115]:
df["allophones"] = ""
df["allophones"] = df["allophones"].apply(lambda x: [])

In [22]:
def batch_predict(model, dataloader):
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader):
            words, _, words_pad_mask, _ = batch
            predictions = model.words_to_allophones(words, 50)
            
            # Конвертируем в numpy и сохраняем
            for i in range(words.size(1)):
                word_seq = words[:, i].cpu().numpy()
                pred_seq = predictions[:, i].cpu().numpy()
                all_predictions.append((word_seq, pred_seq))

            gc.collect()
            torch.cuda.empty_cache()
    
    return all_predictions

In [117]:
ds = WordsDataset(df)
dl = torch.utils.data.DataLoader(ds, batch_size=256, collate_fn=WordsPadder(loaded_word_tokens_encoder.tokens_to_ids["<PAD>"]), shuffle=False)

100%|███████████████████████████████████████████████████████████████████████| 239400/239400 [00:04<00:00, 58762.46it/s]


In [119]:
pred_list = batch_predict(loaded_model, dl)

pred_df = pd.DataFrame(pred_list, columns=["content", "allophones"])

  0%|                                                                                          | 0/936 [00:00<?, ?it/s]C:\python_venv\venv1_py12\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████| 936/936 [31:23<00:00,  2.01s/it]


In [121]:
pred_df["content"] = loaded_word_tokens_encoder.inverse_transform(pred_df["content"])
pred_df["allophones"] = loaded_allophones_tokens_encoder.inverse_transform(pred_df["allophones"])

In [123]:
remove_bos_eos(pred_df, "content")
remove_bos_eos(pred_df, "allophones")

In [124]:
remove_pad(pred_df, "content")
remove_pad(pred_df, "allophones")

In [127]:
df = df.reset_index()

Добавим пунктуацию и паузы:

In [129]:
pred_df["wav_name"] = df["wav_name"]
pred_df["punctuation"] = df["punctuation"]

In [139]:
pred_df["allophones"] += pred_df["punctuation"].apply(list)

In [23]:
def get_pause(punctuation):
    if punctuation == ',':
        return [' ', '-']
    
    if punctuation == '.' or punctuation == '?' or punctuation == '!':
        return [' ', '-', '-']

    return [' ']

In [145]:
pred_df["pause"] = pred_df["punctuation"].apply(get_pause)
pred_df["allophones"] += pred_df["pause"]

In [147]:
pred_df = pred_df.groupby("wav_name").agg({"allophones": "sum"}).reset_index()

И преобразуем полученные тексты в последовательности чисел:

In [157]:
new_allophones_encoder = TokensEncoder()
new_allophones_encoder.fit(pred_df["allophones"])
pred_df["allophones"] = new_allophones_encoder.transform(pred_df["allophones"])

In [156]:
with open("new_allophones_encoder.pickle", "wb") as file:
    pickle.dump(new_allophones_encoder, file)

In [158]:
pred_df.to_csv("data/tokenized_allophones_metadata_RUSLAN_22200.csv", sep='|', index=False, header=False)

## Обучение Tacotron2

Загрузим датасет:

In [24]:
pred_df = pd.read_csv("data/tokenized_allophones_metadata_RUSLAN_22200.csv", delimiter='|', header=None)
pred_df

,0,1
0,000000_RUSLAN,"[4, 5, 5, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,000001_RUSLAN,"[33, 29, 34, 27, 8, 9, 12, 35, 36, 12, 11, 12,..."
2,000002_RUSLAN,"[19, 10, 29, 12, 6, 7, 9, 38, 29, 5, 19, 42, 3..."
3,000003_RUSLAN,"[40, 29, 5, 5, 33, 29, 11, 9, 12, 29, 26, 5, 3..."
4,000004_RUSLAN,"[25, 12, 26, 5, 6, 27, 6, 7, 9, 27, 38, 4, 27,..."
...,...,...
21369,022195_RUSLAN,"[18, 46, 5, 5, 33, 12, 11, 7, 9, 15, 27, 15, 4..."
21370,022196_RUSLAN,"[11, 12, 13, 27, 39, 20, 26, 17, 11, 35, 8, 6,..."
21371,022197_RUSLAN,"[27, 16, 5, 5, 12, 6, 7, 8, 9, 16, 29, 19, 12,..."
21372,022198_RUSLAN,"[40, 29, 38, 5, 24, 36, 42, 18, 54, 9, 32, 13,..."


Загрузим модель и токенизаторы:

In [25]:
loaded_model = torch.load("model.pt", weights_only=False)
loaded_model.eval()
loaded_model = loaded_model.to(device)

with open("word_tokens_encoder.pickle", "rb") as file:
    loaded_word_tokens_encoder = pickle.load(file)

with open("allophones_tokens_encoder.pickle", "rb") as file:
    loaded_allophones_tokens_encoder = pickle.load(file)

with open("new_allophones_encoder.pickle", "rb") as file:
    loaded_new_allophones_encoder = pickle.load(file)

Зададим специальный класс, последовательно применяющий токенизаторы и Seq2seq модель:

In [26]:
def transform_word(word, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder):
    word = word.lower()
    tokenized_word = word_tokens_encoder.transform_one(add_bos_eos_to_list(list(word)))
    tokenized_allophones = model.word_to_allophones(torch.LongTensor(tokenized_word).to(device).unsqueeze(1))
    allophones = remove_bos_eos_from_list(allophones_tokens_encoder.inverse_transform_one(tokenized_allophones.flatten().cpu().tolist()))
    return new_allophones_encoder.transform_one(allophones)

In [27]:
def transform_sentence(sentence, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder):
    sentence = re.sub("[^а-яА-Я,.!?]", " ", sentence)
    words = re.sub(" +", " " , sentence.lower()).split()
    punctuations = [split_letters_and_symbols(word)[1] for word in words]
    words = [split_letters_and_symbols(word)[0] for word in words]
    
    allophones = [transform_word(word, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder) for word in words]
    pauses = [new_allophones_encoder.transform_one(get_pause(punctuation)) for punctuation in punctuations]
    punctuations = [new_allophones_encoder.transform_one(punctuation) for punctuation in punctuations]

    sentence = []
    for i in range(len(allophones)):
        sentence += allophones[i]
        sentence += punctuations[i]
        sentence += pauses[i]

    return sentence

In [28]:
class SpecialTokenizer():
    def __init__(self, word_tokens_encoder, model, allophones_tokens_encoder, new_allophones_encoder):
        self.word_tokens_encoder = word_tokens_encoder
        self.model = model
        self.allophones_tokens_encoder = allophones_tokens_encoder
        self.new_allophones_encoder = new_allophones_encoder
        
        self.tokens_count = 4
        self.tokens_to_ids = {0: 0, 1: 1, 2: 2, 3: 3}
        self.pad_id = 0
        self.use_phonemes = False
        
    
    def fit(self, df_column):
        for index, row in df_column.items():
            for token in row:
                if token not in self.tokens_to_ids:
                    self.tokens_to_ids[token] = self.tokens_count
                    self.tokens_count += 1

        self.ids_to_tokens = {value: key for key, value in self.tokens_to_ids.items()}
        self.characters = Characters(self.tokens_count)

    def transform(self, df_column):
        return df_column.apply(lambda x: [self.tokens_to_ids[token] if token in self.tokens_to_ids else self.tokens_to_ids["<UNK>"] for token in x])

    def inverse_transform(self, df_column):
        return df_column.apply(lambda x: [self.ids_to_tokens[id] for id in x])

    def text_to_ids(self, text, language=None):
        if type(text) == list:
            return text
        
        if text[0] == '[':
            return ast.literal_eval(text)

        return transform_sentence(text, self.word_tokens_encoder, self.model, self.allophones_tokens_encoder, self.new_allophones_encoder)
        
    def print_logs(self, log_level):
        pass

Будем обучать модель с помощью библиотеки Coqui-TTS. Зададим конфигурацию датасета для модели:

In [29]:
 dataset_config = BaseDatasetConfig(
        formatter="ruslan", meta_file_train="tokenized_allophones_metadata_RUSLAN_22200.csv", path="./data/"
    )

Передадим информацию о наборе представленных символов специальному токенизатору:

In [30]:
tokenizer = SpecialTokenizer(loaded_word_tokens_encoder, loaded_model, loaded_allophones_tokens_encoder, loaded_new_allophones_encoder)
tokenizer.fit(pred_df[1].progress_apply(pd.eval))

100%|███████████████████████████████████████████████████████████████████████████| 21374/21374 [00:46<00:00, 464.49it/s]


In [31]:
tokenizer.text_to_ids("Привет!")

C:\python_venv\venv1_py12\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


[38, 11, 12, 43, 12, 10, 11, 12, 10, 59, 9, 32, 32]

In [32]:
tokenizer.text_to_ids("Привет!")

C:\python_venv\venv1_py12\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


[38, 11, 12, 43, 12, 10, 11, 12, 10, 59, 9, 32, 32]

Зададим первую конфигурацию. Эта конфигурация использовалась в начале обучения (более недели):

In [32]:
config = Tacotron2Config(
        batch_size=64,
        eval_batch_size=16,
        max_decoder_steps=1000,
        num_loader_workers=0,
        num_eval_loader_workers=0,
        run_eval=False,
        test_delay_epochs=-1,
        epochs=1000,
        print_step=25,
        print_eval=False,
        mixed_precision=True,
        output_path="train_data1",
        datasets=[dataset_config],
         test_sentences=[
                "Мне потребовалось довольно много времени, чтобы развить голос, и теперь я не собираюсь молчать.",
                "Будь голосом, а не эхом.",
                "Мне жаль, Дэйв. Боюсь, я не могу этого сделать.",
                "Этот пирог просто великолепен. Он такой вкусный и сочный.",
                "До ноября года.",
            ]
    )

    # INITIALIZE THE AUDIO PROCESSOR
    # Audio processor is used for feature extraction and audio I/O.
    # It mainly serves to the dataloader and the training loggers.
ap = AudioProcessor.init_from_config(config)

    # LOAD DATA SAMPLES
    # Each sample is a list of ```[text, audio_file_path, speaker_name]```
    # You can define your custom sample loader returning the list of samples.
    # Or define your custom formatter and pass it to the `load_tts_samples`.
    # Check `TTS.tts.datasets.load_tts_samples` for more details.
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_size=0.1,
)

    # INITIALIZE THE MODEL
    # Models take a config object and a speaker manager as input
    # Config defines the details of the model like the number of layers, the size of the embedding, etc.
    # Speaker manager is used by multi-speaker models.
model = Tacotron2(config, ap, tokenizer, speaker_manager=None)

    # INITIALIZE THE TRAINER
    # Trainer provides a generic API to train all the 🐸TTS models with all its perks like mixed-precision training,
    # distributed training, etc.


Зададим вторую конфигурацию. Эта конфигурация использовалась для продолжения обучения (неделя):

In [33]:
audio_config = BaseAudioConfig(
    sample_rate=16000,
    do_trim_silence=False,
    trim_db=60.0,
    signal_norm=False,
    mel_fmin=0.0,
    mel_fmax=8000,
    spec_gain=1.0,
    log_func="np.log",
    ref_level_db=20,
    preemphasis=0.0,
)

config = Tacotron2Config(
    audio=audio_config,
    run_eval=True,
    test_delay_epochs=-1,
    ga_alpha=0.0,              # Это веса функций потерь такотрона
    decoder_loss_alpha=0.25,   # Это веса функций потерь такотрона
    postnet_loss_alpha=0.25,   # Это веса функций потерь такотрона
    postnet_diff_spec_alpha=0, # Это веса функций потерь такотрона
    decoder_diff_spec_alpha=0, # Это веса функций потерь такотрона
    decoder_ssim_alpha=0,      # Это веса функций потерь такотрона
    postnet_ssim_alpha=0,      # Это веса функций потерь такотрона
    optimizer='Adam',          
    grad_clip=1.0,
    r=2,                       # Это количество фреймов, предсказываемых за раз; меньше -- быстрее сходится!
    max_decoder_steps=500,     # Вот это нужно понижать, чтобы инференс был быстрее
    attention_type="original", # Вот это минорная вариация в архитектуре Такотрона
    double_decoder_consistency=False,     # Это тоже
    epochs=1000,
    batch_size=128,
    eval_batch_size=32,
    lr=1e-5,                              # LR поменьше, чтобы поглубэе спустилося
    lr_scheduler=None,                    # По умолчанию там странный шедулер, чтобы не экспериментировать, лучше старый добрый конст
    print_step=25,
    print_eval=True,
    mixed_precision=False,
    datasets=[dataset_config],
    max_audio_len=160000,                # Чтобы не падало по памяти,если слишком длинная вавка попадает в бач.
    test_sentences=[
                "Мне потребовалось довольно много времени, чтобы развить голос, и теперь я не собираюсь молчать.",
                "Будь голосом, а не эхом.",
                "Мне жаль, Дэйв. Боюсь, я не могу этого сделать.",
                "Этот пирог просто великолепен. Он такой вкусный и сочный.",
                "До ноября года.",
            ]
)

ap = AudioProcessor.init_from_config(config)

    # LOAD DATA SAMPLES
    # Each sample is a list of ```[text, audio_file_path, speaker_name]```
    # You can define your custom sample loader returning the list of samples.
    # Or define your custom formatter and pass it to the `load_tts_samples`.
    # Check `TTS.tts.datasets.load_tts_samples` for more details.
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_size=0.1,
)

    # INITIALIZE THE MODEL
    # Models take a config object and a speaker manager as input
    # Config defines the details of the model like the number of layers, the size of the embedding, etc.
    # Speaker manager is used by multi-speaker models.
model = Tacotron2(config, ap, tokenizer, speaker_manager=None)

Эта строчка необходима для продолжения прерванного обучения (в силу смены конфига или непредвиденного выключения компьютера):

In [35]:
model.load_checkpoint(config, "train_data\\run-December-27-2025_12+35AM-0000000\\best_model.pth")

Обучим модель Tacotron2:

In [ ]:
trainer = Trainer(
    TrainerArgs(), config, "train_data", model=model, train_samples=train_samples, eval_samples=eval_samples
)

trainer.fit()

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 20
 | > Num. of Torch Threads: 20
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=train_data\run-December-27-2025_08+25PM-0000000

 > Model has 28272242 parameters

 > EPOCH: 0/999
 --> train_data\run-December-27-2025_08+25PM-0000000
C:\ИТМО\Магистратура\3-й семестр\Синтез речи\coqui-ai-TTS\TTS\tts\datasets\dataset.py:60: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will b


   --> TIME: 2025-12-27 20:26:14 -- STEP: 25/150 -- GLOBAL_STEP: 25
     | > current_lr: 1e-05  (1.0000000000000003e-05)
     | > decoder_loss: 0.5001177787780762  (0.4911201858520508)
     | > postnet_loss: 0.4230765998363495  (0.4218352198600769)
     | > stopnet_loss: 0.006265060044825077  (0.009978792592883107)
     | > loss: 0.23706366121768951  (0.23821764528751374)
     | > align_error: 0.6173176765441895  (0.6714037597179412)
     | > grad_norm: tensor(0.1759, device='cuda:0')  (tensor(0.2391, device='cuda:0'))
     | > step_time: 0.9896  (0.7097829055786132)
     | > loader_time: 0.9862  (0.8513325309753418)


   --> TIME: 2025-12-27 20:27:09 -- STEP: 50/150 -- GLOBAL_STEP: 50
     | > current_lr: 1e-05  (1.0000000000000011e-05)
     | > decoder_loss: 0.514593780040741  (0.4984502530097961)
     | > postnet_loss: 0.42627421021461487  (0.4228312003612518)
     | > stopnet_loss: 0.004554674029350281  (0.007789051383733747)
     | > loss: 0.23977167904376984  (0.2381094166636467


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.37681374044129345 (+0.0)
     | > avg_decoder_loss: 0.45456383354736096 (+0.0)
     | > avg_postnet_loss: 0.3083062388680199 (+0.0)
     | > avg_stopnet_loss: 0.004569064776384921 (+0.0)
     | > avg_loss: 0.19528658349405636 (+0.0)
     | > avg_align_error: 0.5725436526717561 (+0.0)

 > BEST MODEL : train_data\run-December-27-2025_08+25PM-0000000\best_model_151.pth

 > EPOCH: 1/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-27 20:35:07) 

   --> TIME: 2025-12-27 20:35:45 -- STEP: 24/150 -- GLOBAL_STEP: 175
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.5009249448776245  (0.4933937614162763)
     | > postnet_loss: 0.42490047216415405  (0.4247812206546466)
     | > stopnet_loss: 0.0063040959648787975  (0.01004706604483848)
     | > loss: 0.23776045441627502  (0.2395908087491989)
     | > align_error: 0.6948051452636719  (0.6745311617851257)
     | > grad_norm: tensor(0.1636, device='cuda:0'


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.3467786275979244 (-0.03003511284336907)
     | > avg_decoder_loss: 0.4533551118590615 (-0.0012087216882994412)
     | > avg_postnet_loss: 0.30780844796787604 (-0.0004977909001438463)
     | > avg_stopnet_loss: 0.0045737412398342385 (+4.67646344931745e-06)
     | > avg_loss: 0.19486463137648322 (-0.0004219521175731489)
     | > avg_align_error: 0.5729509181145466 (+0.00040726544279046095)

 > BEST MODEL : train_data\run-December-27-2025_08+25PM-0000000\best_model_302.pth

 > EPOCH: 2/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-27 20:44:17) 

   --> TIME: 2025-12-27 20:44:54 -- STEP: 23/150 -- GLOBAL_STEP: 325
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.49350231885910034  (0.49244445044061413)
     | > postnet_loss: 0.42097553610801697  (0.4240266836207846)
     | > stopnet_loss: 0.006776721682399511  (0.010058640299931814)
     | > loss: 0.23539617657661438  (0.2391764249490655)
     


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.3500558752002138 (+0.0032772476022894237)
     | > avg_decoder_loss: 0.45392566287156305 (+0.0005705510125015301)
     | > avg_postnet_loss: 0.3076583273483046 (-0.00015012061957142553)
     | > avg_stopnet_loss: 0.004567252218045972 (-6.489021788266651e-06)
     | > avg_loss: 0.19496324929324063 (+9.861791675741527e-05)
     | > avg_align_error: 0.5726440110892963 (-0.00030690702525026925)


 > EPOCH: 3/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-27 20:53:32) 

   --> TIME: 2025-12-27 20:54:04 -- STEP: 22/150 -- GLOBAL_STEP: 475
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.5006921887397766  (0.4911060373891484)
     | > postnet_loss: 0.4231713116168976  (0.423049810257825)
     | > stopnet_loss: 0.0067317187786102295  (0.010464683598415419)
     | > loss: 0.23769760131835938  (0.23900364474816757)
     | > align_error: 0.6325793266296387  (0.674554933201183)
     | > grad_norm: tenso

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.2694864814931696 (-0.0779841885422215)
     | > avg_decoder_loss: 0.43546710366552527 (-0.0007764701590394218)
     | > avg_postnet_loss: 0.300639825336861 (+0.00016476981567625781)
     | > avg_stopnet_loss: 0.004302925073936809 (-6.492422526755207e-06)
     | > avg_loss: 0.18832965624151807 (-0.00015941758950552276)
     | > avg_align_error: 0.5717918412251906 (+0.000991624864665086)


 > EPOCH: 173/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-28 22:50:42) 

   --> TIME: 2025-12-28 22:50:45 -- STEP: 2/150 -- GLOBAL_STEP: 26125
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.44308045506477356  (0.44982099533081055)
     | > postnet_loss: 0.3820793032646179  (0.38689084351062775)
     | > stopnet_loss: 0.014646647498011589  (0.014237707480788231)
     | > loss: 0.2209365963935852  (0.22341567277908325)
     | > align_error: 0.7321706116199493  (0.7229353338479996)
     | > grad_norm: tens


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.39437573606317694 (+0.12488925457000732)
     | > avg_decoder_loss: 0.435101435491533 (-0.0003656681739922507)
     | > avg_postnet_loss: 0.3020328472961079 (+0.0013930219592468984)
     | > avg_stopnet_loss: 0.004283162101990346 (-1.9762971946462975e-05)
     | > avg_loss: 0.18856673371611218 (+0.0002370774745941162)
     | > avg_align_error: 0.5705416757952082 (-0.0012501654299823528)


 > EPOCH: 174/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-28 23:00:21) 

   --> TIME: 2025-12-28 23:00:23 -- STEP: 1/150 -- GLOBAL_STEP: 26275
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.4561575949192047  (0.4561575949192047)
     | > postnet_loss: 0.3921721279621124  (0.3921721279621124)
     | > stopnet_loss: 0.014787222258746624  (0.014787222258746624)
     | > loss: 0.22686965763568878  (0.22686965763568878)
     | > align_error: 0.713146299123764  (0.713146299123764)
     | > grad_norm: tensor(


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.4167059840578021 (+0.022330247994625185)
     | > avg_decoder_loss: 0.43409678520578326 (-0.001004650285749753)
     | > avg_postnet_loss: 0.3004055352825107 (-0.0016273120135972263)
     | > avg_stopnet_loss: 0.00429745694749396 (+1.4294845503613936e-05)
     | > avg_loss: 0.18792303741881342 (-0.0006436962972987681)
     | > avg_align_error: 0.5709911928032384 (+0.000449517008030198)

 > BEST MODEL : train_data\run-December-27-2025_08+25PM-0000000\best_model_26425.pth

 > EPOCH: 175/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-28 23:10:06) 

   --> TIME: 2025-12-28 23:10:07 -- STEP: 0/150 -- GLOBAL_STEP: 26425
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.4441450536251068  (0.4441450536251068)
     | > postnet_loss: 0.3824188709259033  (0.3824188709259033)
     | > stopnet_loss: 0.018924783915281296  (0.018924783915281296)
     | > loss: 0.22556577622890472  (0.22556577622890472)
    


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.38706069281606964 (-0.029645291241732485)
     | > avg_decoder_loss: 0.4355922237490163 (+0.0014954385432330297)
     | > avg_postnet_loss: 0.3008454961307121 (+0.000439960848201415)
     | > avg_stopnet_loss: 0.0043205942104881006 (+2.3137262994140685e-05)
     | > avg_loss: 0.1884300232385144 (+0.0005069858197009891)
     | > avg_align_error: 0.5706505314870314 (-0.0003406613162070249)


 > EPOCH: 176/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-28 23:19:47) 

   --> TIME: 2025-12-28 23:20:20 -- STEP: 24/150 -- GLOBAL_STEP: 26600
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.471405953168869  (0.46104441583156586)
     | > postnet_loss: 0.395737886428833  (0.3895092122256756)
     | > stopnet_loss: 0.006128084380179644  (0.009790782545072338)
     | > loss: 0.22291405498981476  (0.22242918921013674)
     | > align_error: 0.6912891864776611  (0.6709716046849886)
     | > grad_norm: tens


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.37994635105133057 (-0.007114341764739074)
     | > avg_decoder_loss: 0.43524411636771576 (-0.0003481073813005331)
     | > avg_postnet_loss: 0.2996396314014089 (-0.0012058647293031743)
     | > avg_stopnet_loss: 0.004325590710240332 (+4.9964997522310844e-06)
     | > avg_loss: 0.1880465274055799 (-0.0003834958329345106)
     | > avg_align_error: 0.5707409806323774 (+9.044914534606097e-05)


 > EPOCH: 177/999
 --> train_data\run-December-27-2025_08+25PM-0000000

 > TRAINING (2025-12-28 23:29:40) 

   --> TIME: 2025-12-28 23:30:13 -- STEP: 23/150 -- GLOBAL_STEP: 26750
     | > current_lr: 1e-05  (1e-05)
     | > decoder_loss: 0.4674304127693176  (0.4604331617769988)
     | > postnet_loss: 0.3927381932735443  (0.38899026357609295)
     | > stopnet_loss: 0.006720614619553089  (0.00976475605579174)
     | > loss: 0.2217627614736557  (0.22212061350760254)
     | > align_error: 0.637677013874054  (0.6713669416697128)
     | > grad_norm: tens